#  [전처리] 상품 이미지 배경 제거 (BiRefNet) 파이프라인

**목적:**
- DB에 적재된 상품들의 원본 이미지(URL)를 다운로드하고, AI(BiRefNet)를 활용해 배경을 깔끔하게 제거(누끼)하여 저장
- 이후 진행될 Jina CLIP 하이브리드 임베딩 추출을 위한 가장 기초적이고 필수적인 전처리 단계

 입력 (Input):
- `products.csv`
- 필수 컬럼: `id` (상품 고유번호), `image_url` (원본 이미지 링크)
- 이 파일은 코랩 로컬이나 구글 드라이브에 업로드

**출력 (Output):**
- `processed_images.zip` (배경이 제거된 .png 파일들이 하나로 압축된 최종 결과물)


**⚙️ 주요 세팅:**
- 배경 제거 AI 모델: `rembg`의 `birefnet-general` (가구, 데스크테리어 소품 등 복잡한 경계선 처리에 최적화)

In [ ]:
# 셀 1: 패키지 설치 (NumPy는 안전하게 두고, SciPy만 업데이트, 필수 엔진 추가)
!pip install -U scipy scikit-image
!pip install rembg sentence-transformers onnxruntime-gpu

import torch
print("torch:", torch.__version__)
print("gpu available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 63.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 77.6 MB/s eta 0:00:00
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.3
    Uninstalling scipy-1.16.3:
      Successfully uninstalled scipy-1.16.3
  Attempting uninstall: scikit-image
    Found existing installation: scikit-image 0.25.2
    Uninstalling scikit-image-0.25.2:
      Successfully uninstalled scikit-image-0.25.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cucim-cu12 26.2.0 requires scikit-image<0.26.0,>=0.19.0, but you have scikit-image 0.26.0 which is incompatible.
INFO: pip is looking at multiple versions of numba to determine which version is compatible with other requirements. This could take a while.
  

torch: 2.10.0+cu128
gpu available: True
gpu: Tesla T4


In [ ]:
# 셀 2: CSV 준비 (업로드 또는 Drive 경로)
import os
CSV_PATH = "products_to_embed.csv"

mode = input("입력 방식 선택 (1: 업로드, 2: Drive 경로) [기본 1]: ").strip() or "1"

if mode == "2":
    from google.colab import drive
    drive.mount("/content/drive")
    CSV_PATH = input("CSV 전체 경로 입력: ").strip()
else:
    from google.colab import files
    uploaded = files.upload()
    print("업로드:", list(uploaded.keys()))
    if "products_to_embed.csv" in uploaded:
        CSV_PATH = "products_to_embed.csv"
    elif len(uploaded) == 1:
        CSV_PATH = list(uploaded.keys())[0]

print("CSV_PATH =", CSV_PATH)
if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(f"CSV 파일을 찾을 수 없습니다: {CSV_PATH}")

입력 방식 선택 (1: 업로드, 2: Drive 경로) [기본 1]: 1


Saving product.csv to product.csv
업로드: ['product.csv']
CSV_PATH = product.csv


In [ ]:
# 셀 3: 공통 설정 및 데이터 로드
import csv
import io
import random
import time
import requests
import gc
import torch
import numpy as np
import pandas as pd
from PIL import Image
from getpass import getpass

HF_TOKEN = getpass("HF_TOKEN 입력 (숨김): ").strip()

MODEL_NAME = "jinaai/jina-clip-v1"
SKIP_REMBG = {"MOUSEPAD", "DESK_SHELF"}
NOISE_SEEDS = [42, 43, 44]
NOISE_DROP_RATES = [0.2, 0.4, 0.6]
TOP_K = [1, 3, 5]
WEIGHT_STEPS = [round(x, 1) for x in np.arange(0.0, 1.01, 0.1)]

with open(CSV_PATH, encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

if not rows:
    raise ValueError("CSV가 비어 있습니다.")

required = {"id", "image_url", "category", "title"}
if not required.issubset(set(rows[0].keys())):
    raise ValueError(f"CSV 컬럼 오류: 필요 {required}, 현재 {set(rows[0].keys())}")

print(f"CSV rows: {len(rows)}")

HF_TOKEN 입력 (숨김): ··········
CSV rows: 2185


In [ ]:
# 셀 4: 전처리 & 즉시 저장 (메모리 최적화 아키텍처)
from rembg import new_session, remove
import io
import os
import time
import requests
from PIL import Image
import gc
import torch

SAVE_DIR = "processed_images"
os.makedirs(SAVE_DIR, exist_ok=True)

def fetch_image_with_retry(url: str, retries: int = 3, timeout: int = 20) -> bytes:
    headers = {"User-Agent": "Mozilla/5.0"}
    last_err = None
    for attempt in range(1, retries + 1):
        try:
            resp = requests.get(url, timeout=timeout, headers=headers)
            resp.raise_for_status()
            return resp.content
        except Exception as e:
            last_err = e
            time.sleep(1.2 * attempt)
    raise last_err

def preprocess_image(url: str, skip_rembg: bool, session) -> Image.Image:
    raw = fetch_image_with_retry(url)
    img = Image.open(io.BytesIO(raw)).convert("RGB")

    if not skip_rembg:
        no_bg = remove(img, session=session)
        if "A" in no_bg.getbands():
            alpha = no_bg.split()[3]
            bbox = alpha.getbbox()
            if bbox:
                no_bg = no_bg.crop(bbox)
                alpha = alpha.crop(bbox)
            bg = Image.new("RGB", no_bg.size, (255, 255, 255))
            bg.paste(no_bg, mask=alpha)
            img = bg

    w, h = img.size
    side = max(w, h)
    canvas = Image.new("RGB", (side, side), (255, 255, 255))
    canvas.paste(img, ((side - w) // 2, (side - h) // 2))
    return canvas

print("\n[전처리] 배경 제거 모델(birefnet) 로딩 중...")
rembg_session = new_session("birefnet-general")

# 이미지를 통째로 담던 images = [] 리스트는 삭제하고, 경로를 담을 image_paths = [] 생성
db_ids, titles, image_paths, failed = [], [], [], []
total_count = len(rows)

print(f" 총 {total_count}개 데이터 처리 및 즉시 저장 시작! (안정성 확보됨)")

for i, r in enumerate(rows):
    try:
        pid = str(r["id"])
        img = preprocess_image(r["image_url"], r["category"] in SKIP_REMBG, rembg_session)

        # 1. 즉시 파일로 저장
        file_path = os.path.join(SAVE_DIR, f"{pid}.png")
        img.save(file_path, format="PNG")

        # 2. 리스트에는 무거운 이미지 대신 가벼운 '문자열 경로'만 저장
        db_ids.append(int(pid))
        titles.append(r["title"])
        image_paths.append(file_path)

        # 3. RAM에서 이미지 객체 완전 파기 (메모리 누수 방지)
        del img

        if (i + 1) % 100 == 0:
            print(f"⏳ 진행 중... ({i + 1} / {total_count}) 완료")

    except Exception as e:
        print(f" [{i}번 에러] ID {r.get('id', '?')}: {type(e).__name__} - {e}")
        failed.append((r.get("id", "?"), str(e)))

print(f"\n 전체 전처리 완료: 성공 {len(image_paths)} / 실패 {len(failed)}")

# VRAM 확보를 위해 rembg 완전 소각
print("[메모리 정리] rembg 세션 삭제 및 VRAM 확보 중...")
del rembg_session
gc.collect()
torch.cuda.empty_cache()
print(" GPU/RAM 메모리 청소 완료. (이제 안전하게 임베딩 가능)")


[전처리] 배경 제거 모델(birefnet) 로딩 중...


  0%|                                               | 0.00/973M [00:00<?, ?B/s]

🚀 총 2185개 데이터 처리 및 즉시 저장 시작! (안정성 확보됨)


/usr/local/lib/python3.12/dist-packages/rembg/sessions/birefnet_general.py:18: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-mat))


⏳ 진행 중... (100 / 2185) 완료
⏳ 진행 중... (200 / 2185) 완료
⏳ 진행 중... (300 / 2185) 완료
⏳ 진행 중... (400 / 2185) 완료
⏳ 진행 중... (500 / 2185) 완료
⏳ 진행 중... (600 / 2185) 완료
⏳ 진행 중... (700 / 2185) 완료
⏳ 진행 중... (800 / 2185) 완료
⏳ 진행 중... (900 / 2185) 완료
⏳ 진행 중... (1000 / 2185) 완료
⏳ 진행 중... (1100 / 2185) 완료
⏳ 진행 중... (1200 / 2185) 완료
⏳ 진행 중... (1300 / 2185) 완료
⏳ 진행 중... (1400 / 2185) 완료
⏳ 진행 중... (1500 / 2185) 완료
⏳ 진행 중... (1600 / 2185) 완료
⏳ 진행 중... (1700 / 2185) 완료
⏳ 진행 중... (1800 / 2185) 완료
⏳ 진행 중... (1900 / 2185) 완료
⏳ 진행 중... (2000 / 2185) 완료
⏳ 진행 중... (2100 / 2185) 완료

✅ 전체 전처리 완료: 성공 2185 / 실패 0
[메모리 정리] rembg 세션 삭제 및 VRAM 확보 중...
✅ GPU/RAM 메모리 청소 완료. (이제 안전하게 임베딩 가능)


In [ ]:
# [셀 4.5] 이미 디스크에 저장된 폴더를 압축하여 다운로드
import shutil
from google.colab import files

SAVE_DIR = "processed_images"

print("압축 파일(processed_images.zip) 생성 중...")
shutil.make_archive("processed_images", 'zip', SAVE_DIR)

print("내 컴퓨터로 다운로드 시작")
files.download("processed_images.zip")

📦 압축 파일(processed_images.zip) 생성 중...
내 컴퓨터로 다운로드 시작


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>